[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/00-eda.ipynb)

In [ ]:
%%capture
!pip install transformers datasets seqeval accelerate torch numpy
!pip install --upgrade transformers

In [ ]:
import os
# Suppress all tqdm/HuggingFace progress bars before any library is imported
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
)

In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [ ]:
# ==========================================
# 1. Load Dataset & Extract Labels
# ==========================================
# MedMentions-ZS is available directly on the Hub
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")

# The dataset stores tags as strings (e.g., 'B-T062', 'I-T062', 'O').
# We need to extract all unique tags to create our label mappings.
# Note: Adjust the column names 'tokens' and 'ner_tags' if the dataset uses different keys.
unique_tags = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        unique_tags.update(row)

label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# ==========================================
# 2. Tokenization & Label Alignment
# ==========================================
model_id = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_and_align_labels(examples):
    # Tokenize the pre-split words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )

    labels = []
    for i, label_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens (like [CLS] and [SEP]) map to None. We set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_sequence[word_idx]])
            # For subsequent tokens of the same word, also assign -100 (or you can assign the I- tag).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply to train, validation, and test splits
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
# ==========================================
# 3. Exploratory Data Analysis
# ==========================================

# Flatten all tags into a single list
all_tags = [tag for sublist in dataset['train']['ner_tags'] for tag in sublist]

# Create a frequency table
tag_counts = pd.Series(all_tags).value_counts()

# Separate 'O' from entities
o_count = tag_counts.get('O', 0)
entity_count = tag_counts.sum() - o_count

print(f"Number of unique tags: {len(tag_counts)}")
print(f"Total 'O' tags: {o_count}")
print(f"Total Entity tags: {entity_count}")
print(f"Percentage of 'O' tags: {o_count/tag_counts.sum()*100:.2f}%")

In [ ]:
# Table for the 10 most frequent tags, excluding 'O'
top_10_tags = tag_counts.drop('O').head(10)
print(top_10_tags)

In [ ]:
# Distribution of Token Sequence Lengths
seq_lengths = [len(tokens) for tokens in dataset['train']['tokens']]
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(seq_lengths, bins=50, kde=True, color='purple', ax=ax)
ax.set_title('Distribution of Token Sequence Lengths')
ax.set_xlabel('Number of Tokens')
ax.set_ylabel('Frequency')
ax.axvline(np.percentile(seq_lengths, 95), color='red', linestyle='dashed', label='95th Percentile')
ax.legend()
plt.tight_layout()
plt.savefig('ner_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_length_distribution.png")

# Tag Frequency Bar Chart (sorted descending, log scale, excluding 'O')
tag_counts_without_o = tag_counts.drop('O', errors='ignore').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(tag_counts_without_o)), tag_counts_without_o.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(tag_counts_without_o)))
ax.set_xticklabels(tag_counts_without_o.index, rotation=45, ha='right', fontsize=8)
ax.set_yscale('log')
ax.set_xlabel('Entity Tag')
ax.set_ylabel('Count (log scale)')
ax.set_title('NER Entity Tag Frequency (excluding O), sorted by count descending')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ner_tag_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_tag_frequency.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate Exact Class Weights for Training
all_tags_flat = [tag for sublist in dataset['train']['ner_tags'] for tag in sublist]
tag_counts = pd.Series(all_tags_flat).value_counts()

num_classes = len(label_list)
total_samples = tag_counts.sum()

# Formula: Total Samples / (Number of Classes * Samples per Class)
class_weights_dict = {
    tag: total_samples / (num_classes * count)
    for tag, count in tag_counts.items()
}

# Map to the label2id order so we can pass it directly to PyTorch
class_weights = [class_weights_dict.get(id2label[i], 1.0) for i in range(num_classes)]

print("Top 5 highest weights (Rarest classes):", sorted(class_weights, reverse=True)[:5])
print("Weight for 'O' class:", class_weights_dict.get('O', 1.0))

# 2. Entity Support Bucket Analysis (excluding 'O')
entity_counts = tag_counts.drop('O', errors='ignore')

def support_bucket(count):
    if count == 1:
        return '1'
    elif count <= 10:
        return '2-10'
    elif count <= 100:
        return '11-100'
    elif count <= 1000:
        return '101-1000'
    else:
        return '1000+'

buckets = entity_counts.apply(support_bucket)
bucket_order = ['1', '2-10', '11-100', '101-1000', '1000+']
bucket_counts = buckets.value_counts().reindex(bucket_order, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
bars = ax.bar(bucket_order, bucket_counts.values, color=colors, alpha=0.85, edgecolor='white')
for bar, cnt in zip(bars, bucket_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(cnt), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel('Support (Token Count in Train Split)', fontsize=12)
ax.set_ylabel('Number of Entity Types', fontsize=12)
ax.set_title('NER Entity Types by Support Bucket\nMedMentions MTI881 — Training Set', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ner_entity_support_buckets.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_entity_support_buckets.png")
print("\nEntity types per support bucket:")
for bucket, cnt in bucket_counts.items():
    print(f"  [{bucket:>8}] tokens: {cnt} entity types")

In [ ]:
# NER: Token Length Distribution by Split
fig, ax = plt.subplots(figsize=(12, 5))
split_colors = {'train': 'steelblue', 'validation': 'darkorange', 'test': 'green'}
for split, color in split_colors.items():
    if split in dataset:
        lengths = [len(tokens) for tokens in dataset[split]['tokens']]
        ax.hist(lengths, bins=50, alpha=0.5, color=color, label=split)
        for pct, ls in [(50, ':'), (90, '--'), (99, '-.')]:
            val = np.percentile(lengths, pct)
            ax.axvline(val, color=color, linestyle=ls, alpha=0.8, linewidth=1.2)
ax.axvline(512, color='red', linestyle='-', linewidth=2, alpha=0.8, label='BERT max_length=512')
ax.set_xlabel('Tokens per Sequence')
ax.set_ylabel('Count')
ax.set_title('NER (MedMentions): Token Length Distribution by Split\n(dotted=p50, dashed=p90, dash-dot=p99)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ner_length_by_split.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_length_by_split.png")

# Print percentile table
print("\nToken length percentiles:")
print(f"{'Split':<12} {'p50':>6} {'p90':>6} {'p99':>6} {'max':>6}")
for split in ['train', 'validation', 'test']:
    if split in dataset:
        lengths = [len(tokens) for tokens in dataset[split]['tokens']]
        print(f"{split:<12} {int(np.percentile(lengths,50)):>6} {int(np.percentile(lengths,90)):>6} {int(np.percentile(lengths,99)):>6} {max(lengths):>6}")

In [ ]:
# NER: Entity Type Co-occurrence Heatmap
from collections import defaultdict
import itertools
import seaborn as sns
import numpy as np

# ner_tags are strings like 'B-T103', 'I-T103', 'O'
# Get top entity tags by frequency (excluding 'O'), sorted descending
entity_tag_counts = tag_counts.drop('O', errors='ignore').sort_values(ascending=False)

def root_type(tag):
    """Strip B-/I- prefix to get the UMLS type code."""
    return tag[2:] if tag.startswith(('B-', 'I-')) else tag

# Collect top root types (deduplicated, preserving frequency order)
top_roots = list(dict.fromkeys(root_type(t) for t in entity_tag_counts.index))[:12]

# Count co-occurrences (per sentence in training set)
cooccur = defaultdict(int)
for tags_seq in dataset['train']['ner_tags']:
    # tags_seq is already a list of strings
    roots_in_sentence = set(root_type(t) for t in tags_seq if t != 'O')
    roots_in_sentence = roots_in_sentence & set(top_roots)
    for a, b in itertools.combinations(sorted(roots_in_sentence), 2):
        cooccur[(a, b)] += 1
    for a in roots_in_sentence:
        cooccur[(a, a)] += 1

cooccur_matrix = np.zeros((len(top_roots), len(top_roots)))
for i, a in enumerate(top_roots):
    for j, b in enumerate(top_roots):
        key = (min(a, b), max(a, b)) if a != b else (a, a)
        cooccur_matrix[i, j] = cooccur[key]

log_matrix = np.log1p(cooccur_matrix)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(log_matrix, xticklabels=top_roots, yticklabels=top_roots,
            cmap='YlOrRd', ax=ax, annot=cooccur_matrix.astype(int), fmt='d',
            annot_kws={'size': 8})
ax.set_title('Entity Type Co-occurrence (log scale counts)\nMedMentions NER Training Set — Top 12 UMLS Entity Types')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('ner_entity_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_entity_cooccurrence.png")

---
## MedQA-USMLE-4-Options Dataset EDA

In [ ]:
# Load MedQA Dataset
from datasets import load_dataset
import pandas as pd
import numpy as np

qa_dataset = load_dataset("GBaker/MedQA-USMLE-4-options")
print("MedQA-USMLE-4-options splits:", {k: len(v) for k, v in qa_dataset.items()})
print("\nFeatures:", qa_dataset['train'].features)
print("\nSample record:")
sample = qa_dataset['train'][0]
for k, v in sample.items():
    v_str = str(v)[:120]
    print(f"  {k}: {v_str}")

In [ ]:
# QA: Answer Option Distribution (is there label bias?)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

option_names = ['A', 'B', 'C', 'D']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, split in zip(axes, ['train', 'test']):
    if split not in qa_dataset:
        ax.set_visible(False)
        continue
    # answer_idx is a string: 'A', 'B', 'C', or 'D'
    answer_letters = qa_dataset[split]['answer_idx']
    counts = pd.Series(answer_letters).value_counts().reindex(option_names, fill_value=0)
    ax.bar(option_names, counts.values, color=['steelblue', 'darkorange', 'green', 'red'], alpha=0.8)
    ax.axhline(y=len(answer_letters) / 4, color='black', linestyle='--', linewidth=1.5, label='Random baseline (uniform)')
    ax.set_xlabel('Correct Answer Option')
    ax.set_ylabel('Count')
    ax.set_title(f'Answer Distribution — {split} split\n(n={len(answer_letters)})')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    for i, (opt, cnt) in enumerate(zip(option_names, counts.values)):
        ax.text(i, cnt + 5, f'{cnt / len(answer_letters) * 100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('MedQA-USMLE: Answer Option Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('qa_answer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: qa_answer_distribution.png")

In [ ]:
# QA: Question Length Distribution
import matplotlib.pyplot as plt
import numpy as np

train_q_lengths = [len(ex['question'].split()) for ex in qa_dataset['train']]
train_opt_lengths = []
for ex in qa_dataset['train']:
    opts = ex.get('options', {})
    if isinstance(opts, dict):
        combined = ex['question'] + ' ' + ' '.join(opts.values())
    else:
        combined = ex['question']
    train_opt_lengths.append(len(combined.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_q_lengths, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
for pct, ls, label in [(50, '--', 'p50'), (90, '-.', 'p90'), (95, ':', 'p95')]:
    val = int(np.percentile(train_q_lengths, pct))
    axes[0].axvline(val, color='red', linestyle=ls, alpha=0.8, label=f'{label}={val}')
axes[0].set_xlabel('Words in Question')
axes[0].set_ylabel('Count')
axes[0].set_title('Question Length Distribution (words)\nMedQA-USMLE Train Split')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(train_opt_lengths, bins=50, color='darkorange', alpha=0.8, edgecolor='white')
bert_512_approx = 350  # ~512 tokens ≈ 350-380 words for clinical text
axes[1].axvline(bert_512_approx, color='red', linestyle='-', linewidth=2, alpha=0.8,
                label=f'Approx. BERT 512-token limit (~{bert_512_approx} words)')
for pct, ls, label in [(50, '--', 'p50'), (90, '-.', 'p90'), (95, ':', 'p95')]:
    val = int(np.percentile(train_opt_lengths, pct))
    axes[1].axvline(val, color='navy', linestyle=ls, alpha=0.8, label=f'{label}={val}')
axes[1].set_xlabel('Words (question + all options combined)')
axes[1].set_ylabel('Count')
axes[1].set_title('Combined Input Length Distribution (words)\nMedQA-USMLE Train Split')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('MedQA-USMLE Input Length Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('qa_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: qa_length_distribution.png")

# Summary stats
print("\nQuestion length (words):")
print(f"  Mean: {np.mean(train_q_lengths):.1f}, Median: {np.median(train_q_lengths):.1f}, "
      f"p95: {np.percentile(train_q_lengths, 95):.0f}, Max: {max(train_q_lengths)}")
print(f"\nCombined input length (question + options, words):")
print(f"  Mean: {np.mean(train_opt_lengths):.1f}, Median: {np.median(train_opt_lengths):.1f}, "
      f"p95: {np.percentile(train_opt_lengths, 95):.0f}")
pct_truncated = sum(1 for l in train_opt_lengths if l > bert_512_approx) / len(train_opt_lengths) * 100
print(f"  Estimated % inputs exceeding BERT 512-token limit: {pct_truncated:.1f}%")